# model_experiment_XGBoost

მიზანი: იგივე end-to-end feature pipeline გამოვიყენოთ XGBoost-ზე და შევადაროთ LightGBM-ს WMAE validation-ით.

In [1]:
# Run once if packages are missing:
# !pip install -r requirements.txt

In [2]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

import pandas as pd
import numpy as np
import joblib
import wandb

from src.config import WANDB_ENTITY, WANDB_PROJECT, MODELS_DIR, SUBMISSIONS_DIR, VALIDATION_FOLDS, VALIDATION_WEEKS, RANDOM_STATE
from src.data_loading import load_raw_data, describe_dataframes, make_submission_frame
from src.validation import evaluate_time_folds, summarize_cv
from src.metrics import wmae
from src.wandb_utils import safe_wandb_init, save_and_log_model, log_dataframe_as_artifact

pd.set_option('display.max_columns', 100)
MODELS_DIR.mkdir(exist_ok=True, parents=True)
SUBMISSIONS_DIR.mkdir(exist_ok=True, parents=True)
print('W&B entity:', WANDB_ENTITY)
print('W&B project:', WANDB_PROJECT)

from src.models import WalmartSalesForecaster, make_xgboost_model

W&B entity: gchal22-free-university-of-tbilisi-
W&B project: store_sales_forecast


In [3]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\West\_netrc.


wandb: Currently logged in as: gchal22 (gchal22-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
data = load_raw_data()
train = data['train'].copy()
features = data['features'].copy()
stores = data['stores'].copy()
test = data['test'].copy()
sample_submission = data['sample_submission'].copy()
describe_dataframes(data)

,name,rows,columns,column_names
0,train,421570,5,"Store, Dept, Date, Weekly_Sales, IsHoliday"
1,test,115064,4,"Store, Dept, Date, IsHoliday"
2,features,8190,12,"Store, Date, Temperature, Fuel_Price, MarkDown..."
3,stores,45,3,"Store, Type, Size"
4,sample_submission,115064,2,"Id, Weekly_Sales"


In [5]:
FAST_DEV_RUN = False
if FAST_DEV_RUN:
    train_for_cv = train[train['Store'].isin([1, 2, 3])].copy()
else:
    train_for_cv = train.copy()

xgboost_params = {
    'n_estimators': 900,
    'learning_rate': 0.035,
    'max_depth': 8,
    'min_child_weight': 5,
    'subsample': 0.85,
    'colsample_bytree': 0.85,
    'reg_alpha': 0.05,
    'reg_lambda': 0.5,
}

def forecaster_factory():
    model = make_xgboost_model(random_state=RANDOM_STATE, **xgboost_params)
    return WalmartSalesForecaster(model=model, model_name='XGBoost')

In [6]:
with safe_wandb_init(
    run_name='XGBoost_CV',
    group='XGBoost',
    job_type='cross_validation',
    config={
        'model': 'XGBoost',
        'validation_folds': VALIDATION_FOLDS,
        'validation_weeks': VALIDATION_WEEKS,
        'fast_dev_run': FAST_DEV_RUN,
        **xgboost_params,
    },
) as run:
    cv_results = evaluate_time_folds(
        forecaster_factory,
        train_for_cv,
        features,
        stores,
        n_folds=VALIDATION_FOLDS,
        validation_weeks=VALIDATION_WEEKS,
        log_callback=wandb.log,
    )
    summary = summarize_cv(cv_results)
    wandb.log(summary)
    wandb.log({'cv_results': wandb.Table(dataframe=cv_results)})
    log_dataframe_as_artifact(run, cv_results, 'xgboost-cv-results', 'validation_results', 'reports/xgboost_cv_results.csv')

cv_results

USING FIXED evaluate_time_folds v2


cv_mae_mean,▁
cv_wmae_mean,▁
cv_wmae_std,▁
fold_1_fold,▁
fold_1_mae,▁
fold_1_train_rows,▁
fold_1_valid_rows,▁
fold_1_wmae,▁
fold_2_fold,▁
fold_2_mae,▁
+11,...


,fold,train_start,train_end,valid_start,valid_end,train_rows,valid_rows,wmae,mae,holiday_wmae_part
0,1,2010-02-05,2012-05-11,2012-05-18,2012-07-06,350595,23608,1547.618475,1547.618475,NaN
1,2,2010-02-05,2012-07-06,2012-07-13,2012-08-31,374203,23638,1547.693402,1547.693402,NaN
2,3,2010-02-05,2012-08-31,2012-09-07,2012-10-26,397841,23729,1385.678602,1381.375313,1394.285543


In [7]:
summary

{'cv_wmae_mean': 1493.6634928516735,
 'cv_wmae_std': 76.35685461570988,
 'cv_mae_mean': 1492.2290632364122}

In [8]:
REGISTER_AS_BEST = False  # Change to True only if this beats LightGBM on CV/Kaggle.

final_model = WalmartSalesForecaster(
    model=make_xgboost_model(random_state=RANDOM_STATE, **xgboost_params),
    model_name='XGBoost',
)

with safe_wandb_init(
    run_name='XGBoost_Final_Model',
    group='XGBoost',
    job_type='final_training',
    config={'model': 'XGBoost', **xgboost_params},
) as run:
    final_model.fit(train, features, stores)
    model_path = MODELS_DIR / 'xgboost_pipeline.joblib'
    metadata = {
        'architecture': 'XGBoost',
        'metric': 'WMAE',
        'uses_raw_test': True,
        'feature_columns': final_model.feature_columns_,
        'notes': 'Recursive lag/rolling feature pipeline trained on full train.csv',
    }
    save_and_log_model(run, final_model, 'xgboost-pipeline', model_path, metadata=metadata, aliases=['latest', 'candidate'])
    if REGISTER_AS_BEST:
        save_and_log_model(run, final_model, 'best-model', model_path, metadata=metadata, aliases=['latest', 'production'])

In [9]:
preds = final_model.predict(test)
submission = make_submission_frame(test, preds, sample_submission)
submission_path = SUBMISSIONS_DIR / 'submission_xgboost.csv'
submission.to_csv(submission_path, index=False)
submission.head(), submission_path

(               Id  Weekly_Sales
 0  1_1_2012-11-02  35829.074219
 1  1_1_2012-11-09  25417.667969
 2  1_1_2012-11-16  22664.527344
 3  1_1_2012-11-23  22652.544922
 4  1_1_2012-11-30  26466.169922,
 WindowsPath('C:/walmart/submissions/submission_xgboost.csv'))

In [10]:
with safe_wandb_init('XGBoost_Submission', 'XGBoost', 'submission') as run:
    log_dataframe_as_artifact(run, submission, 'xgboost-submission', 'submission', submission_path)
    wandb.log({
        'prediction_min': float(submission['Weekly_Sales'].min()),
        'prediction_max': float(submission['Weekly_Sales'].max()),
        'prediction_mean': float(submission['Weekly_Sales'].mean()),
    })

prediction_max,▁
prediction_mean,▁
prediction_min,▁
prediction_max,261135.14062
prediction_mean,16884.66005
prediction_min,0
